In [1]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2) Setup paths
from pathlib import Path
from zipfile import ZipFile
import os

# Source ZIPs in Drive
DRIVE = Path("/content/drive/MyDrive/deepfake_project")
ZIP_TRAIN = DRIVE / "zips/dataset_train.zip"
ZIP_TEST  = DRIVE / "zips/dataset_test.zip"

# Destination folders in Colab
DST_AFHQ_TRAIN   = Path("/content/stargan-v2/combined_balanced/train")
DST_AFHQ_TEST    = Path("/content/stargan-v2/combined_balanced/test")



# Create destinations
DST_AFHQ_TRAIN.mkdir(parents=True, exist_ok=True)
DST_AFHQ_TEST.mkdir(parents=True, exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def safe_extract_images(zip_path: Path, dest_dir: Path):
    """
    Extract only image files from the zip to dest_dir (preserves internal subpaths),
    with path traversal protection.
    """
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP not found: {zip_path}")

    extracted = 0
    with ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            # Skip directories
            if info.is_dir():
                continue
            # Only extract images
            suffix = Path(info.filename).suffix.lower()
            if suffix not in IMG_EXT:
                continue

            # Path traversal protection
            target_path = dest_dir / info.filename
            target_path_parent = target_path.parent.resolve()
            dest_root_resolved = dest_dir.resolve()
            if not str(target_path_parent).startswith(str(dest_root_resolved)):
                # Skip anything that tries to escape the destination
                continue

            # Make sure subdirs exist
            target_path.parent.mkdir(parents=True, exist_ok=True)

            # Extract this single file
            with zf.open(info, "r") as src_f, open(target_path, "wb") as dst_f:
                dst_f.write(src_f.read())
            extracted += 1
    return extracted

def count_images(root: Path) -> int:
    return sum(1 for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT)

# 3) Extract train + test images into /content/stargan-v2/combined_balanced
n_train = safe_extract_images(ZIP_TRAIN, DST_AFHQ_TRAIN)
n_test  = safe_extract_images(ZIP_TEST,  DST_AFHQ_TEST)


print(f"[DONE] Extracted TRAIN images -> {DST_AFHQ_TRAIN}: {n_train} files")
print(f"[DONE] Extracted TEST  images -> {DST_AFHQ_TEST}: {n_test} files")

Mounted at /content/drive
[DONE] Extracted TRAIN images -> /content/stargan-v2/combined_balanced/train: 23040 files
[DONE] Extracted TEST  images -> /content/stargan-v2/combined_balanced/test: 11520 files


In [2]:
!pip install shap lime --quiet

import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
import numpy as np
import os

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 25.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
EXPLAIN_DIR = "/content/stargan-v2/explainability"
os.makedirs(EXPLAIN_DIR, exist_ok=True)

In [4]:
from sklearn.metrics import roc_auc_score

def explain_xgb_with_shap(
    model,
    X_train,
    X_test,
    y_test,
    feature_names,
    prefix,
    max_bg=1000,
    max_test=2000,
    random_state=42
):
    """
    model         : fitted XGBoost (or other tree) model
    X_train       : numpy array or DataFrame (train features)
    X_test        : numpy array or DataFrame (test features)
    y_test        : test labels (0/1)
    feature_names : list of feature names (len == X.shape[1])
    prefix        : short string for filenames, e.g. "rq1_forensic"
    """
    print(f"\n=== SHAP explainability for {prefix} ===")

    # Ensure numpy arrays
    if isinstance(X_train, pd.DataFrame):
        X_train_np = X_train.values
    else:
        X_train_np = X_train

    if isinstance(X_test, pd.DataFrame):
        X_test_np = X_test.values
    else:
        X_test_np = X_test

    n_bg   = min(max_bg, X_train_np.shape[0])
    n_test = min(max_test, X_test_np.shape[0])

    rng = np.random.RandomState(random_state)
    bg_idx   = rng.choice(X_train_np.shape[0], size=n_bg, replace=False)
    test_idx = rng.choice(X_test_np.shape[0], size=n_test, replace=False)

    X_bg   = X_train_np[bg_idx]
    X_tsub = X_test_np[test_idx]
    y_tsub = np.array(y_test)[test_idx]

    # TreeExplainer is optimized for XGBoost / tree ensembles
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_tsub)

    # 1) Global summary (beeswarm)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values,
        X_tsub,
        feature_names=feature_names,
        show=False
    )
    out_path = os.path.join(EXPLAIN_DIR, f"{prefix}_shap_summary_beeswarm.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"  - Saved SHAP beeswarm to {out_path}")

    # 2) Global bar plot of mean |SHAP|
    plt.figure(figsize=(8, 6))
    shap.summary_plot(
        shap_values,
        X_tsub,
        feature_names=feature_names,
        plot_type="bar",
        show=False
    )
    out_path = os.path.join(EXPLAIN_DIR, f"{prefix}_shap_importance_bar.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    print(f"  - Saved SHAP bar plot to {out_path}")

    # 3) Save mean |SHAP| as CSV for tables in the dissertation
    abs_mean = np.mean(np.abs(shap_values), axis=0)
    shap_df = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": abs_mean,
    }).sort_values("mean_abs_shap", ascending=False)
    out_csv = os.path.join(EXPLAIN_DIR, f"{prefix}_shap_importance.csv")
    shap_df.to_csv(out_csv, index=False)
    print(f"  - Saved SHAP importance CSV to {out_csv}")

    # 4) Optional: local explanations for a few test points
    # pick one TP, one FP, one FN if possible
    y_pred_proba = model.predict_proba(X_test_np)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    idx_tp = np.where((y_pred == 1) & (y_test == 1))[0]
    idx_fp = np.where((y_pred == 1) & (y_test == 0))[0]
    idx_fn = np.where((y_pred == 0) & (y_test == 1))[0]

    def _local_plot(idx, tag):
        x = X_test_np[idx:idx+1]
        shap_val = explainer.shap_values(x)
        plt.figure(figsize=(8, 4))
        shap.force_plot(
            explainer.expected_value,
            shap_val,
            x,
            feature_names=feature_names,
            matplotlib=True,
            show=False
        )
        out_path = os.path.join(EXPLAIN_DIR, f"{prefix}_local_{tag}_idx{idx}.png")
        plt.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"  - Saved local SHAP force plot ({tag}) to {out_path}")

    if len(idx_tp) > 0:
        _local_plot(idx_tp[0], "tp")
    if len(idx_fp) > 0:
        _local_plot(idx_fp[0], "fp")
    if len(idx_fn) > 0:
        _local_plot(idx_fn[0], "fn")

    return shap_df


In [5]:
ZIP_BOVW = DRIVE / "output/bovw3/stargan-v2_data_bovw_kmeans_pca-20251214-124333.zip"
DST_BOVW   = Path("/content/stargan-v2/bovw")
DST_BOVW.mkdir(parents=True, exist_ok=True)

def safe_extract_files(zip_path: Path, dest_dir: Path):
    """
    Extract only image files from the zip to dest_dir (preserves internal subpaths),
    with path traversal protection.
    """
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP not found: {zip_path}")

    extracted = 0
    with ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            # Skip directories
            if info.is_dir():
                continue


            # Path traversal protection
            target_path = dest_dir / info.filename
            target_path_parent = target_path.parent.resolve()
            dest_root_resolved = dest_dir.resolve()
            if not str(target_path_parent).startswith(str(dest_root_resolved)):
                # Skip anything that tries to escape the destination
                continue

            # Make sure subdirs exist
            target_path.parent.mkdir(parents=True, exist_ok=True)

            # Extract this single file
            with zf.open(info, "r") as src_f, open(target_path, "wb") as dst_f:
                dst_f.write(src_f.read())
            extracted += 1
    return extracted

n_bovw = safe_extract_files(ZIP_BOVW, DST_BOVW)
print(f"[DONE] Extracted BOVW files -> {DST_BOVW}: {n_bovw} files")

[DONE] Extracted BOVW files -> /content/stargan-v2/bovw: 23094 files


In [6]:
!pip install -q xgboost

import time
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_recall_fscore_support
)
from xgboost import XGBClassifier

In [7]:
BOVW_DIR = Path("/content/stargan-v2/bovw/bovw")

train_df = pd.read_csv(BOVW_DIR / "image_features.csv")
test_df  = pd.read_csv(BOVW_DIR / "image_features_test.csv")

print(train_df.shape, test_df.shape)
train_df.head()

(23040, 402) (11520, 402)


,path,label,label_name,species,fake_technique,width,height,aspect_ratio,brightness,contrast,...,prob_k50_40,prob_k50_41,prob_k50_42,prob_k50_43,prob_k50_44,prob_k50_45,prob_k50_46,prob_k50_47,prob_k50_48,prob_k50_49
0,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,78.899395,57.872423,...,0.015301,0.025958,0.028789,0.016578,0.018042,0.016342,0.016605,0.018252,0.019138,0.019481
1,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,93.176861,73.581886,...,0.014565,0.021546,0.030138,0.016381,0.018243,0.016636,0.017337,0.019808,0.016841,0.021363
2,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,161.828793,63.737348,...,0.015897,0.016159,0.028673,0.019622,0.023557,0.021720,0.021713,0.030435,0.016491,0.016366
3,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,139.004795,59.327718,...,0.023926,0.023850,0.015993,0.015539,0.020943,0.014498,0.018323,0.015481,0.020297,0.023937
4,/content/stargan-v2/combined_balanced/train/re...,0,real,cat,none,512,512,1.0,66.725933,47.867575,...,0.016417,0.016702,0.023671,0.024368,0.018516,0.024236,0.017439,0.023641,0.021530,0.015731


In [8]:
y_train = train_df["label"].astype(int).values
y_test  = test_df["label"].astype(int).values


In [9]:
from collections import defaultdict

def run_xgb_cv_and_test(X_train, y_train, X_test, y_test, model_name,
                        n_splits=5, random_state=42):
    """
    5-fold CV on TRAIN with XGBoost, then fit on full TRAIN and evaluate on TEST.
    Returns:
      final_model, oof_proba, test_proba, cv_summary, test_metrics
    """
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test  = np.asarray(X_test,  dtype=np.float32)
    y_train = np.asarray(y_train, dtype=int)
    y_test  = np.asarray(y_test,  dtype=int)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_proba = np.zeros(len(y_train), dtype=float)
    fold_stats = []

    print(f"\n=== {model_name}: 5-fold CV ===")
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        model = XGBClassifier(
            n_estimators=1000,
            max_depth=10,
            learning_rate=0.001,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=random_state
        )

        t0 = time.time()
        model.fit(X_tr, y_tr)
        train_time = time.time() - t0

        val_proba = model.predict_proba(X_val)[:, 1]
        oof_proba[val_idx] = val_proba

        auc = roc_auc_score(y_val, val_proba)
        val_pred = (val_proba >= 0.5).astype(int)
        acc = accuracy_score(y_val, val_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_val, val_pred, average="binary", zero_division=0
        )

        fold_stats.append({
            "fold": fold,
            "auc": auc,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "train_time_s": train_time
        })

        print(f"  Fold {fold}: AUC={auc:.3f}, F1={f1:.3f}, Acc={acc:.3f}, "
              f"Prec={prec:.3f}, Rec={rec:.3f}, time={train_time:.2f}s")

    # CV summary
    cv_summary = {}
    for key in ["auc", "accuracy", "precision", "recall", "f1", "train_time_s"]:
        vals = [fs[key] for fs in fold_stats]
        cv_summary[f"{key}_mean"] = float(np.mean(vals))
        cv_summary[f"{key}_std"]  = float(np.std(vals))

    print("\nCV summary:")
    for k, v in cv_summary.items():
        print(f"  {k}: {v:.4f}")

    # Fit final model on full TRAIN and evaluate on TEST
    final_model = XGBClassifier(
        n_estimators=1000,
        max_depth=10,
        learning_rate=0.001,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=random_state
    )

    t0 = time.time()
    final_model.fit(X_train, y_train)
    full_train_time = time.time() - t0

    test_proba = final_model.predict_proba(X_test)[:, 1]
    test_pred  = (test_proba >= 0.5).astype(int)

    test_auc = roc_auc_score(y_test, test_proba)
    test_acc = accuracy_score(y_test, test_pred)
    test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(
        y_test, test_pred, average="binary", zero_division=0
    )

    test_metrics = {
        "auc": float(test_auc),
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "f1": float(test_f1),
        "full_train_time_s": float(full_train_time),
        "n_features": int(X_train.shape[1])
    }

    print(f"\n=== {model_name}: TEST metrics ===")
    print(f"  AUC={test_auc:.3f}, F1={test_f1:.3f}, Acc={test_acc:.3f}, "
          f"Prec={test_prec:.3f}, Rec={test_rec:.3f}")
    print(f"  Features={X_train.shape[1]}, Train_time_full={full_train_time:.2f}s")

    return final_model, oof_proba, test_proba, cv_summary, test_metrics


In [10]:
forensic_cols = [
    "brightness", "contrast", "sharpness_l1_mean",
    "edge_density", "lap_var", "file_size"
]

X_train_forensic = train_df[forensic_cols].values
X_test_forensic  = test_df[forensic_cols].values

forensic_model, forensic_oof, forensic_test_proba, forensic_cv, forensic_test = \
    run_xgb_cv_and_test(X_train_forensic, y_train,
                        X_test_forensic, y_test,
                        model_name="RQ1_Forensic_XGB")

print("\nRQ1 forensic TEST metrics:", forensic_test)



=== RQ1_Forensic_XGB: 5-fold CV ===
  Fold 1: AUC=0.930, F1=0.846, Acc=0.840, Prec=0.816, Rec=0.878, time=16.75s
  Fold 2: AUC=0.932, F1=0.841, Acc=0.834, Prec=0.808, Rec=0.877, time=11.48s
  Fold 3: AUC=0.931, F1=0.842, Acc=0.836, Prec=0.810, Rec=0.877, time=2.32s
  Fold 4: AUC=0.930, F1=0.839, Acc=0.831, Prec=0.802, Rec=0.880, time=5.10s
  Fold 5: AUC=0.929, F1=0.839, Acc=0.832, Prec=0.803, Rec=0.879, time=2.29s

CV summary:
  auc_mean: 0.9304
  auc_std: 0.0009
  accuracy_mean: 0.8346
  accuracy_std: 0.0031
  precision_mean: 0.8079
  precision_std: 0.0049
  recall_mean: 0.8781
  recall_std: 0.0011
  f1_mean: 0.8415
  f1_std: 0.0024
  train_time_s_mean: 7.5864
  train_time_s_std: 5.6770

=== RQ1_Forensic_XGB: TEST metrics ===
  AUC=0.932, F1=0.845, Acc=0.838, Prec=0.812, Rec=0.880
  Features=6, Train_time_full=2.56s

RQ1 forensic TEST metrics: {'auc': 0.9321120726032023, 'accuracy': 0.8383680555555556, 'precision': 0.8124398845783906, 'recall': 0.8798611111111111, 'f1': 0.84480746791

In [11]:
# Example: RQ1 forensic model
# Assume:
#   xgb_forensic_full : fitted XGB model
#   X_forensic_train, X_forensic_test
#   y_train, y_test
#   forensic_cols: list of forensic feature column names

shap_rq1 = explain_xgb_with_shap(
    model=forensic_model,
    X_train=X_train_forensic,
    X_test=X_test_forensic,
    y_test=y_test,
    feature_names=forensic_cols,
    prefix="rq1_forensic_xgb"
)


=== SHAP explainability for rq1_forensic_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq1_forensic_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq1_forensic_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq1_forensic_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq1_forensic_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq1_forensic_xgb_local_fp_idx17.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq1_forensic_xgb_local_fn_idx6725.png


<Figure size 800x400 with 0 Axes>

<Figure size 800x400 with 0 Axes>

<Figure size 800x400 with 0 Axes>

In [12]:
# grab all BoVW columns
bovw_cols = [c for c in train_df.columns if c.startswith("bovw_")]
print("BoVW dims:", len(bovw_cols))

# soft cluster probabilities (if present)
prob_k2_cols  = [c for c in train_df.columns if c.startswith("prob_k2_")]
prob_k10_cols = [c for c in train_df.columns if c.startswith("prob_k10_")]
prob_k50_cols = [c for c in train_df.columns if c.startswith("prob_k50_")]

print("prob_k2 dims:", len(prob_k2_cols),
      "prob_k10 dims:", len(prob_k10_cols),
      "prob_k50 dims:", len(prob_k50_cols))

bovw_k_feats = bovw_cols + prob_k2_cols + prob_k10_cols + prob_k50_cols

X_train_bovw_k = train_df[bovw_k_feats].values
X_test_bovw_k  = test_df[bovw_k_feats].values

bovw_k_model, bovw_k_oof, bovw_k_test_proba, bovw_k_cv, bovw_k_test = \
    run_xgb_cv_and_test(X_train_bovw_k, y_train,
                        X_test_bovw_k, y_test,
                        model_name="RQ2_BoVW_plus_Kmeans_XGB")

print("\nRQ2 BoVW+K TEST metrics:", bovw_k_test)


BoVW dims: 257
prob_k2 dims: 2 prob_k10 dims: 10 prob_k50 dims: 50

=== RQ2_BoVW_plus_Kmeans_XGB: 5-fold CV ===
  Fold 1: AUC=0.563, F1=0.492, Acc=0.542, Prec=0.553, Rec=0.443, time=154.70s
  Fold 2: AUC=0.580, F1=0.514, Acc=0.559, Prec=0.572, Rec=0.467, time=147.00s
  Fold 3: AUC=0.581, F1=0.498, Acc=0.556, Prec=0.573, Rec=0.440, time=148.09s
  Fold 4: AUC=0.586, F1=0.508, Acc=0.555, Prec=0.568, Rec=0.459, time=154.28s
  Fold 5: AUC=0.577, F1=0.504, Acc=0.554, Prec=0.569, Rec=0.452, time=149.39s

CV summary:
  auc_mean: 0.5774
  auc_std: 0.0079
  accuracy_mean: 0.5533
  accuracy_std: 0.0058
  precision_mean: 0.5668
  precision_std: 0.0074
  recall_mean: 0.4523
  recall_std: 0.0102
  f1_mean: 0.5030
  f1_std: 0.0079
  train_time_s_mean: 150.6940
  train_time_s_std: 3.1932

=== RQ2_BoVW_plus_Kmeans_XGB: TEST metrics ===
  AUC=0.637, F1=0.530, Acc=0.591, Prec=0.623, Rec=0.461
  Features=319, Train_time_full=160.79s

RQ2 BoVW+K TEST metrics: {'auc': 0.6370944854359568, 'accuracy': 0.59123

In [13]:
# RQ2: BoVW+KMeans model
shap_rq2 = explain_xgb_with_shap(
    model=bovw_k_model,
    X_train=X_train_bovw_k,
    X_test=X_test_bovw_k,
    y_test=y_test,
    feature_names=bovw_k_feats,  # e.g. ["bovw_0", ..., "prob_k2_0", "prob_k10_0", ...]
    prefix="rq2_bovw_k_xgb"
)


=== SHAP explainability for rq2_bovw_k_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq2_bovw_k_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq2_bovw_k_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq2_bovw_k_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq2_bovw_k_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq2_bovw_k_xgb_local_fp_idx2.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq2_bovw_k_xgb_local_fn_idx5767.png


In [14]:
sil_df = pd.read_csv(BOVW_DIR / "silhouette_k_grid.csv")
print("Silhouette grid:\n", sil_df)
best_k = sil_df.iloc[sil_df["silhouette"].idxmax()]["k"]
print("Best k by silhouette:", best_k)


Silhouette grid:
     k  silhouette
0   2    0.141861
1   3    0.118226
2   5    0.088363
3  10    0.066255
4  20    0.055312
5  50    0.035830
Best k by silhouette: 2.0


In [15]:
# Raw BoVW only
X_train_bovw = train_df[bovw_cols].values
X_test_bovw  = test_df[bovw_cols].values

bovw_model, bovw_oof, bovw_test_proba, bovw_cv, bovw_test = \
    run_xgb_cv_and_test(X_train_bovw, y_train,
                        X_test_bovw, y_test,
                        model_name="RQ3_BoVW_raw_XGB")

# PCA features only
pca_cols = [c for c in train_df.columns if c.startswith("pca_")]
X_train_pca = train_df[pca_cols].values
X_test_pca  = test_df[pca_cols].values

pca_model, pca_oof, pca_test_proba, pca_cv, pca_test = \
    run_xgb_cv_and_test(X_train_pca, y_train,
                        X_test_pca, y_test,
                        model_name="RQ3_BoVW_PCA_XGB")

print("\nRQ3 BoVW raw TEST metrics:", bovw_test)
print("RQ3 BoVW PCA TEST metrics:", pca_test)



=== RQ3_BoVW_raw_XGB: 5-fold CV ===
  Fold 1: AUC=0.563, F1=0.509, Acc=0.549, Prec=0.558, Rec=0.467, time=129.24s
  Fold 2: AUC=0.577, F1=0.524, Acc=0.554, Prec=0.561, Rec=0.491, time=133.71s
  Fold 3: AUC=0.584, F1=0.514, Acc=0.557, Prec=0.569, Rec=0.469, time=130.98s
  Fold 4: AUC=0.580, F1=0.516, Acc=0.554, Prec=0.564, Rec=0.475, time=132.62s
  Fold 5: AUC=0.575, F1=0.514, Acc=0.561, Prec=0.575, Rec=0.464, time=130.26s

CV summary:
  auc_mean: 0.5758
  auc_std: 0.0071
  accuracy_mean: 0.5549
  accuracy_std: 0.0039
  precision_mean: 0.5657
  precision_std: 0.0060
  recall_mean: 0.4734
  recall_std: 0.0094
  f1_mean: 0.5153
  f1_std: 0.0048
  train_time_s_mean: 131.3618
  train_time_s_std: 1.6105

=== RQ3_BoVW_raw_XGB: TEST metrics ===
  AUC=0.632, F1=0.538, Acc=0.587, Prec=0.610, Rec=0.482
  Features=257, Train_time_full=140.72s

=== RQ3_BoVW_PCA_XGB: 5-fold CV ===
  Fold 1: AUC=0.585, F1=0.504, Acc=0.561, Prec=0.579, Rec=0.446, time=32.81s
  Fold 2: AUC=0.586, F1=0.507, Acc=0.559, 

In [16]:
# RQ3: BoVW-raw
shap_rq3_raw = explain_xgb_with_shap(
    model=bovw_model,
    X_train=X_train_bovw,
    X_test=X_test_bovw,
    y_test=y_test,
    feature_names=bovw_cols,

    prefix="rq3_bovw_raw_xgb"
)


=== SHAP explainability for rq3_bovw_raw_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_local_fp_idx3.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq3_bovw_raw_xgb_local_fn_idx5773.png


In [17]:
# RQ3: BoVW+PCA
shap_rq3_pca = explain_xgb_with_shap(
    model=pca_model,
    X_train=X_train_pca,
    X_test=X_test_pca,
    y_test=y_test,
    feature_names=pca_cols,  # e.g. ["pca_0", ..., "pca_63"]
    prefix="rq3_bovw_pca_xgb"
)


=== SHAP explainability for rq3_bovw_pca_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_local_fp_idx2.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq3_bovw_pca_xgb_local_fn_idx5763.png


In [18]:
import tensorflow as tf
from tensorflow.keras import layers, models
import glob, os

IMG_SIZE = (256, 256)
BATCH_SIZE = 64

def list_images_with_labels(root_dir):
    paths, labels = [], []
    for label_name, label_val in [("real", 0), ("fake", 1)]:
        for p in glob.glob(os.path.join(root_dir, label_name, "**", "*.*"), recursive=True):
            ext = os.path.splitext(p)[1].lower()
            if ext in [".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"]:
                paths.append(p)
                labels.append(label_val)
    return np.array(paths), np.array(labels)

train_img_paths, train_img_labels = list_images_with_labels(str(DST_AFHQ_TRAIN))
test_img_paths,  test_img_labels  = list_images_with_labels(str(DST_AFHQ_TEST))

print("Train images:", len(train_img_paths), "Test images:", len(test_img_paths))


Train images: 23040 Test images: 11520


In [19]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE   = (256, 256)
INPUT_SHAPE = IMG_SIZE + (3,)

def build_small_cnn(input_shape=INPUT_SHAPE):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # --- Optional augmentation (can comment out if you want) ---
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        # -----------------------------------------------------------

        layers.Rescaling(1./255),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.GlobalAveragePooling2D(),

        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    return model



In [20]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 64

def decode_and_resize(path, label):
    # path is a tf.string
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    # model will rescale to [0,1], so keep as uint8/float32 here
    img = tf.cast(img, tf.float32)
    label = tf.cast(label, tf.float32)
    return img, label

def make_dataset(paths, labels, shuffle=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds


In [21]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
import time

def run_cnn_cv_and_test(train_paths, train_labels,
                        test_paths, test_labels,
                        n_splits=5, epochs=16, batch_size=64,
                        random_state=42):

    train_paths = np.array(train_paths)
    train_labels = np.array(train_labels).astype(int)
    test_paths = np.array(test_paths)
    test_labels = np.array(test_labels).astype(int)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_proba = np.zeros(len(train_labels), dtype=float)
    fold_stats = []

    print("\n=== CNN: 5-fold CV ===")
    fold_idx = 0

    for tr_idx, val_idx in skf.split(train_paths, train_labels):
        fold_idx += 1
        print(f"\n--- Fold {fold_idx}/{n_splits} ---")

        tr_paths, val_paths = train_paths[tr_idx], train_paths[val_idx]
        tr_labels, val_labels = train_labels[tr_idx], train_labels[val_idx]

        train_ds = make_dataset(tr_paths, tr_labels, shuffle=True,  batch_size=batch_size)
        val_ds   = make_dataset(val_paths, val_labels, shuffle=False, batch_size=batch_size)

        model = build_small_cnn()

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=2, restore_best_weights=True
            )
        ]

        t0 = time.time()
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=1,
            callbacks=callbacks
        )
        train_time = time.time() - t0

        # Get probabilities on validation set
        val_proba = model.predict(val_ds).ravel()  # sigmoid outputs ∈ [0,1]
        oof_proba[val_idx] = val_proba

        val_pred = (val_proba >= 0.5).astype(int)

        fold_auc = roc_auc_score(val_labels, val_proba)
        fold_acc = accuracy_score(val_labels, val_pred)
        fold_prec, fold_rec, fold_f1, _ = precision_recall_fscore_support(
            val_labels, val_pred, average="binary", zero_division=0
        )

        print(f"Fold {fold_idx} AUC={fold_auc:.3f}, F1={fold_f1:.3f}, "
              f"Acc={fold_acc:.3f}, Prec={fold_prec:.3f}, Rec={fold_rec:.3f}, "
              f"train_time={train_time:.2f}s")

        fold_stats.append({
            "fold": fold_idx,
            "auc": fold_auc,
            "accuracy": fold_acc,
            "precision": fold_prec,
            "recall": fold_rec,
            "f1": fold_f1,
            "train_time_s": train_time
        })

    # CV summary
    cv_summary = {}
    for key in ["auc", "accuracy", "precision", "recall", "f1", "train_time_s"]:
        vals = [fs[key] for fs in fold_stats]
        cv_summary[f"{key}_mean"] = float(np.mean(vals))
        cv_summary[f"{key}_std"]  = float(np.std(vals))

    print("\n=== CNN CV summary ===")
    for k, v in cv_summary.items():
        print(f"  {k}: {v:.4f}")

    # === Train final model on full TRAIN and evaluate on TEST ===
    full_train_ds = make_dataset(train_paths, train_labels, shuffle=True, batch_size=batch_size)
    test_ds       = make_dataset(test_paths,  test_labels,  shuffle=False, batch_size=batch_size)

    final_model = build_small_cnn()

    t0 = time.time()
    final_model.fit(full_train_ds, epochs=epochs, verbose=1)
    full_train_time = time.time() - t0

    # Inference time
    t0 = time.time()
    test_proba = final_model.predict(test_ds).ravel()
    inference_time = time.time() - t0
    time_per_image = inference_time / len(test_labels)

    test_pred = (test_proba >= 0.5).astype(int)

    test_auc = roc_auc_score(test_labels, test_proba)
    test_acc = accuracy_score(test_labels, test_pred)
    test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(
        test_labels, test_pred, average="binary", zero_division=0
    )

    test_metrics = {
        "auc": float(test_auc),
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "f1": float(test_f1),
        "full_train_time_s": float(full_train_time),
        "inference_time_s_total": float(inference_time),
        "inference_time_s_per_image": float(time_per_image),
        "n_params": int(final_model.count_params())
    }

    print("\n=== CNN TEST metrics ===")
    print(f"  AUC={test_auc:.3f}, F1={test_f1:.3f}, Acc={test_acc:.3f}, "
          f"Prec={test_prec:.3f}, Rec={test_rec:.3f}")
    print(f"  Train_time_full={full_train_time:.2f}s, "
          f"Total_infer_time={inference_time:.2f}s, "
          f"Per_image={time_per_image*1000:.2f} ms")
    print(f"  Model parameters: {final_model.count_params()}")

    return final_model, oof_proba, test_proba, cv_summary, test_metrics


In [22]:
cnn_model, cnn_oof_proba, cnn_test_proba, cnn_cv, cnn_test = \
    run_cnn_cv_and_test(train_img_paths, train_img_labels,
                        test_img_paths,  test_img_labels,
                        n_splits=5, epochs=16, batch_size=64)

print("\nCNN CV summary:", cnn_cv)
print("CNN TEST metrics:", cnn_test)



=== CNN: 5-fold CV ===

--- Fold 1/5 ---
Epoch 1/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 36s 47ms/step - accuracy: 0.4966 - auc: 0.4898 - loss: 0.6938 - val_accuracy: 0.4998 - val_auc: 0.5050 - val_loss: 0.6932
Epoch 2/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 27s 44ms/step - accuracy: 0.4967 - auc: 0.4962 - loss: 0.6936 - val_accuracy: 0.5000 - val_auc: 0.5201 - val_loss: 0.6931
Epoch 3/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 26s 43ms/step - accuracy: 0.5051 - auc: 0.5016 - loss: 0.6931 - val_accuracy: 0.5000 - val_auc: 0.5291 - val_loss: 0.6930
Epoch 4/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 26s 44ms/step - accuracy: 0.5054 - auc: 0.5006 - loss: 0.6931 - val_accuracy: 0.5111 - val_auc: 0.5126 - val_loss: 0.6929
Epoch 5/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 26s 43ms/step - accuracy: 0.5050 - auc: 0.5083 - loss: 0.6929 - val_accuracy: 0.5221 - val_auc: 0.5284 - val_loss: 0.6923
Epoch 6/16
288/288 ━━━━━━━━━━━━━━━━━━━━ 26s 43ms/step - accuracy: 0.5074 - auc: 0.5112 - loss: 0.6931 - val_accuracy: 0.5130 - val_auc: 0.5233 - val

In [23]:
import numpy as np
import tensorflow as tf

#train_img_paths, train_img_labels = list_images_with_labels(str(DST_AFHQ_TRAIN))
#test_img_paths,  test_img_labels  = list_images_with_labels(str(DST_AFHQ_TEST))

# Load all test images into memory for LIME (ok for ~10k images at 128x128)
def load_image_raw(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img.set_shape((*IMG_SIZE, 3))
    return img.numpy()  # float32, 0-255

X_test_imgs = np.stack([load_image_raw(p) for p in test_img_paths])
y_test = np.array(test_img_labels, dtype=int)
print("X_test_imgs:", X_test_imgs.shape, "y_test:", y_test.shape)


X_test_imgs: (11520, 256, 256, 3) y_test: (11520,)


In [24]:
def cnn_classifier_fn(images: np.ndarray):
    """
    images: (N, H, W, 3) in 0-255 (uint8 or float).
    Returns probs for shape (N, 2): [p(real), p(fake)].
    """
    imgs = tf.convert_to_tensor(images, dtype=tf.float32)
    preds = cnn_model.predict(imgs, verbose=0).reshape(-1)  # sigmoid -> p(fake)
    preds_fake = preds
    preds_real = 1.0 - preds_fake
    return np.vstack([preds_real, preds_fake]).T


In [25]:
from sklearn.metrics import roc_auc_score

# y_cnn_test_proba, y_cnn_test_pred already computed earlier
print("CNN Test AUC:", roc_auc_score(y_test, cnn_test_proba))

cnn_test_pred_soft = (cnn_test_proba >= 0.1).astype(int)

tp_idx = np.where((y_test == 1) & (cnn_test_proba == 1))[0]
tn_idx = np.where((y_test == 0) & (cnn_test_proba == 1))[0]
fp_idx = np.where((y_test == 0) & (cnn_test_proba == 0))[0]
fn_idx = np.where((y_test == 1) & (cnn_test_proba == 0))[0]


CNN Test AUC: 0.762745044849537


In [26]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
from pathlib import Path

lime_img_explainer = lime_image.LimeImageExplainer()

EXPLAIN_IMG_DIR = Path("/content/stargan-v2/explainability/lime_cnn_images")
EXPLAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

def pick_some(idxs, k=2):
    idxs = np.array(idxs)
    if len(idxs) == 0:
        return []
    if len(idxs) <= k:
        return idxs.tolist()
    return rng.choice(idxs, size=k, replace=False).tolist()

def explain_cnn_image(idx, label_name_map=None):
    # image + meta
    img = X_test_imgs[idx].astype(np.uint8)
    true_label = y_test[idx]
    pred_prob = float(cnn_test_proba[idx])
    pred_label = int(pred_prob >= 0.5)
    path = test_img_paths[idx]
    fake_tech = test_df.iloc[idx]["fake_technique"] if "fake_technique" in test_df.columns else "unknown"
    label_name = test_df.iloc[idx]["label_name"] if "label_name" in test_df.columns else str(true_label)

    print("="*80)
    print(f"CNN LIME for test index {idx}")
    print(f"path          : {path}")
    print(f"true label    : {true_label} ({label_name})")
    print(f"pred label    : {pred_label} ({'fake' if pred_label==1 else 'real'})")
    print(f"pred prob(fake): {pred_prob:.3f}")
    if true_label == 1:
        print(f"fake_technique: {fake_tech}")
    print("-"*80)

    explanation = lime_img_explainer.explain_instance(
        image=img,
        classifier_fn=cnn_classifier_fn,
        top_labels=2,
        hide_color=0,
        num_samples=1000
    )

    # For the "fake" class (1)
    temp, mask = explanation.get_image_and_mask(
        label=1,
        positive_only=True,
        num_features=10,
        hide_rest=False
    )

    # Plot & save overlay
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(img)
    ax[0].set_title("Original")
    ax[0].axis("off")

    ax[1].imshow(mark_boundaries(temp / 255.0, mask))
    ax[1].set_title("LIME – regions supporting 'fake'")
    ax[1].axis("off")

    fname = EXPLAIN_IMG_DIR / f"lime_cnn_idx{idx}_true{true_label}_pred{pred_label}.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=200)
    plt.close()
    print(f"Saved LIME image explanation to {fname}")
    print()


In [27]:
#example_groups = {
#    "TP_fake":        pick_some(tp_idx,  k=2),
#    "TN_real":        pick_some(tn_idx,  k=2),
#    "FP_real_as_fake": pick_some(fp_idx, k=2),
#    "FN_fake_as_real": pick_some(fn_idx, k=2),
#}

#for group_name, idxs in example_groups.items():
#    print(f"\n### {group_name} examples")
#    if not idxs:
#        print("  (none found)")
#        continue
#    for idx in idxs:
#        explain_cnn_image(idx)


In [28]:
import numpy as np

# Flatten proba if needed
scores = cnn_test_proba.ravel()

# Indices of real / fake
idx_real = np.where(y_test == 0)[0]
idx_fake = np.where(y_test == 1)[0]

def top_k(idxs, score_vec, k, reverse=False):
    """Return up to k indices from idxs ordered by score."""
    idxs = np.array(idxs)
    if len(idxs) == 0:
        return []
    order = np.argsort(score_vec[idxs])
    if reverse:  # highest scores first
        order = order[::-1]
    sel = idxs[order[:k]]
    return sel.tolist()

# "Most confident" groups, even if model is bad
TP_like = top_k(idx_fake, scores, k=5, reverse=True)   # true fake, highest P(fake)
FN_like = top_k(idx_fake, scores, k=5, reverse=False)  # true fake, lowest P(fake)
TN_like = top_k(idx_real, scores, k=5, reverse=False)  # true real, lowest P(fake)
FP_like = top_k(idx_real, scores, k=5, reverse=True)   # true real, highest P(fake)

example_groups = {
    "TP_like_fake":        TP_like,
    "TN_like_real":        TN_like,
    "FP_real_as_fake":     FP_like,
    "FN_fake_as_real":     FN_like,
}

for group_name, idxs in example_groups.items():
    print(f"\n### {group_name} examples")
    if not idxs:
        print("  (none found)")
        continue
    for idx in idxs:
        explain_cnn_image(idx)



### TP_like_fake examples
CNN LIME for test index 11473
path          : /content/stargan-v2/combined_balanced/test/fake/swap_like/swap_like__7b9d7364__flickr_wild_001313_swap_30330928__flickr_wild_002242.png
true label    : 1 (fake)
pred label    : 1 (fake)
pred prob(fake): 0.999
fake_technique: swap_like
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx11473_true1_pred1.png

CNN LIME for test index 11254
path          : /content/stargan-v2/combined_balanced/test/fake/swap_like/swap_like__6a3dc63e__pixabay_cat_001162_swap_1c43d58f__pixabay_cat_004186.png
true label    : 1 (fake)
pred label    : 1 (fake)
pred prob(fake): 0.998
fake_technique: swap_like
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx11254_true1_pred1.png

CNN LIME for test index 10953
path          : /content/stargan-v2/combined_balanced/test/fake/swap_like/swap_like__e90d87df__pixabay_wild_001225_swap_5c134357__flickr_wild_002299.png
true label    : 1 (fake)
pred label    : 1 (fake)
pred prob(fake): 0.998
fake_technique: swap_like
--------------------------------------------------------------------------------


...

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx6933_true1_pred0.png

CNN LIME for test index 7609
path          : /content/stargan-v2/combined_balanced/test/fake/inpaint/inpaint__e9158f0e__pixabay_cat_003549_inpaint.png
true label    : 1 (fake)
pred label    : 0 (real)
pred prob(fake): 0.124
fake_technique: copy_move
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx7609_true1_pred0.png

CNN LIME for test index 7661
path          : /content/stargan-v2/combined_balanced/test/fake/inpaint/inpaint__ecb8b20a__flickr_wild_001966_inpaint.png
true label    : 1 (fake)
pred label    : 0 (real)
pred prob(fake): 0.129
fake_technique: copy_move
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_idx7661_true1_pred0.png



In [29]:
# Ensemble

ensemble_X_train = np.vstack([
    forensic_oof,
    bovw_k_oof,
    pca_oof,
    cnn_oof_proba  # when you have it
]).T

ensemble_X_test = np.vstack([
    forensic_test_proba,
    bovw_k_test_proba,
    pca_test_proba,
    cnn_test_proba  # when you have it
]).T

ens_model, ens_oof, ens_test_proba, ens_cv, ens_test = \
    run_xgb_cv_and_test(ensemble_X_train, y_train,
                        ensemble_X_test, y_test,
                        model_name="RQ5_Ensemble_XGB")

print("\nRQ5 ensemble TEST metrics:", ens_test)



=== RQ5_Ensemble_XGB: 5-fold CV ===
  Fold 1: AUC=0.943, F1=0.860, Acc=0.856, Prec=0.839, Rec=0.881, time=1.95s
  Fold 2: AUC=0.928, F1=0.837, Acc=0.830, Prec=0.803, Rec=0.873, time=4.66s
  Fold 3: AUC=0.930, F1=0.848, Acc=0.838, Prec=0.797, Rec=0.907, time=2.00s
  Fold 4: AUC=0.946, F1=0.854, Acc=0.857, Prec=0.874, Rec=0.835, time=1.91s
  Fold 5: AUC=0.926, F1=0.840, Acc=0.830, Prec=0.792, Rec=0.893, time=1.99s

CV summary:
  auc_mean: 0.9346
  auc_std: 0.0083
  accuracy_mean: 0.8421
  accuracy_std: 0.0123
  precision_mean: 0.8212
  precision_std: 0.0312
  recall_mean: 0.8778
  recall_std: 0.0242
  f1_mean: 0.8477
  f1_std: 0.0086
  train_time_s_mean: 2.5019
  train_time_s_std: 1.0775

=== RQ5_Ensemble_XGB: TEST metrics ===
  AUC=0.953, F1=0.861, Acc=0.865, Prec=0.886, Rec=0.837
  Features=4, Train_time_full=2.13s

RQ5 ensemble TEST metrics: {'auc': 0.9533714162567515, 'accuracy': 0.8647569444444444, 'precision': 0.8864974245768947, 'recall': 0.8366319444444444, 'f1': 0.8608431582708

In [30]:
# RQ5: Ensemble (forensic + BoVW + PCA + cluster probs, etc.)
# Example: after you build the proba columns

ensemble_X_train = pd.DataFrame({
    "forensic_proba": forensic_oof,
    "bovw_k_proba":   bovw_k_oof,
    "pca_proba":      pca_oof,
    "cnn_proba":      cnn_oof_proba,
})

ensemble_X_test = pd.DataFrame({
    "forensic_proba": forensic_test_proba,
    "bovw_k_proba":   bovw_k_test_proba,
    "pca_proba":      pca_test_proba,
    "cnn_proba":      cnn_test_proba,
})

shap_rq5_ens = explain_xgb_with_shap(
    model=ens_model,
    X_train=ensemble_X_train,
    X_test=ensemble_X_test,
    y_test=y_test,
    feature_names=list(ensemble_X_train.columns),
    # ["forensic_test_proba","bovw_k_test_proba", "pca_test_proba", "cnn_test_proba"],  # e.g. ["forensic_score","bovw_pca_score","prob_k2_1","prob_k10_4",...]
    prefix="rq5_ensemble_xgb"
)


=== SHAP explainability for rq5_ensemble_xgb ===
  - Saved SHAP beeswarm to /content/stargan-v2/explainability/rq5_ensemble_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/stargan-v2/explainability/rq5_ensemble_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/stargan-v2/explainability/rq5_ensemble_xgb_shap_importance.csv
  - Saved local SHAP force plot (tp) to /content/stargan-v2/explainability/rq5_ensemble_xgb_local_tp_idx5760.png
  - Saved local SHAP force plot (fp) to /content/stargan-v2/explainability/rq5_ensemble_xgb_local_fp_idx13.png
  - Saved local SHAP force plot (fn) to /content/stargan-v2/explainability/rq5_ensemble_xgb_local_fn_idx6720.png


In [31]:
import numpy as np
import pandas as pd

# Where to save lift tables
LIFT_OUT_DIR = DRIVE / "output" / "lift_tables"
LIFT_OUT_DIR.mkdir(parents=True, exist_ok=True)

def compute_lift_table(
    y_true,
    y_score,
    base_df,
    model_name: str,
    n_bins: int = 10,
    out_dir: Path = LIFT_OUT_DIR,
):
    """
    Build a decile-based lift table on the TEST set.

    y_true  : array-like of 0/1 labels (1 = deepfake)
    y_score : array-like of predicted probabilities for class 1 (deepfake)
    base_df : DataFrame containing at least columns: 'label', 'label_name', 'fake_technique'
    model_name : string used in the output CSV file name
    n_bins  : number of bins (10 for deciles)
    out_dir : directory for CSV output

    Returns: lift_df (pandas DataFrame)
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    if y_true.shape[0] != y_score.shape[0]:
        raise ValueError("y_true and y_score must have the same length")

    df = base_df.copy().reset_index(drop=True)
    if len(df) != len(y_true):
        raise ValueError("base_df length must match y_true/y_score length")

    df["y_true"] = y_true
    df["score"] = y_score

    # Sort by descending predicted probability (highest-risk first)
    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    N = len(df)
    df["rank"] = np.arange(1, N + 1)

    # Assign deciles: 1 = top 10% highest score, 10 = lowest 10%
    df["decile"] = ((df["rank"] - 1) / (N / n_bins)).astype(int) + 1
    df.loc[df["decile"] > n_bins, "decile"] = n_bins

    # Overall target rate (deepfake rate) for lift calculations
    base_rate = df["y_true"].mean()
    total_pos = df["y_true"].sum()

    # Basic per-decile stats
    grouped = df.groupby("decile", as_index=False)
    lift_df = grouped["y_true"].agg(
        n="size",
        n_fake="sum",
    )
    lift_df["n_real"] = lift_df["n"] - lift_df["n_fake"]
    lift_df["pct_fake"] = lift_df["n_fake"] / lift_df["n"].replace(0, np.nan)
    lift_df["pct_real"] = lift_df["n_real"] / lift_df["n"].replace(0, np.nan)

    # Cumulative stats and lift
    lift_df["cum_n"] = lift_df["n"].cumsum()
    lift_df["cum_fake"] = lift_df["n_fake"].cumsum()
    lift_df["cum_fake_rate"] = lift_df["cum_fake"] / (total_pos if total_pos > 0 else np.nan)
    lift_df["pop_frac"] = lift_df["cum_n"] / float(N)

    # Per-decile lift vs overall deepfake rate
    lift_df["lift"] = lift_df["pct_fake"] / (base_rate if base_rate > 0 else np.nan)
    # Cumulative lift (how much better than random by this decile)
    lift_df["cum_lift"] = lift_df["cum_fake_rate"] / lift_df["pop_frac"].replace(0, np.nan)

    # Per-deepfake-method breakdown (counts and % among fakes in that decile)
    if "fake_technique" in df.columns:
        # Only look at fakes for technique stats
        fake_only = df[df["y_true"] == 1].copy()
        if not fake_only.empty:
            tech_tab = (
                fake_only
                .pivot_table(
                    index="decile",
                    columns="fake_technique",
                    values="y_true",
                    aggfunc="size",
                    fill_value=0,
                )
            )
            # Counts per technique
            tech_tab_counts = tech_tab.add_prefix("n_fake_")
            tech_tab_counts.reset_index(inplace=True)

            # Merge counts into lift_df
            lift_df = lift_df.merge(tech_tab_counts, on="decile", how="left")
            lift_df.fillna(0, inplace=True)

            # Percent-of-fakes per decile for each technique
            fake_cols = [c for c in lift_df.columns if c.startswith("n_fake_")]
            for c in fake_cols:
                pct_col = c.replace("n_fake_", "pct_fake_")
                lift_df[pct_col] = lift_df[c] / lift_df["n_fake"].replace(0, np.nan)

    # Save to CSV
    out_path = out_dir / f"lift_{model_name}_test.csv"
    lift_df.to_csv(out_path, index=False)
    print(f"[LIFT] Saved {model_name} lift table to: {out_path}")

    return lift_df


In [32]:
# === Build lift tables on TEST for each model ===

# Sanity check that all proba arrays line up with test_df
print("Test length:", len(test_df))
print("  y_test:", len(y_test))
print("  forensic_test_proba:", len(forensic_test_proba))
print("  bovw_k_test_proba:", len(bovw_k_test_proba))
print("  bovw_test_proba:", len(bovw_test_proba))
print("  pca_test_proba:", len(pca_test_proba))
print("  cnn_test_proba:", len(cnn_test_proba))
print("  ens_test_proba:", len(ens_test_proba))

lift_rq1_forensic = compute_lift_table(
    y_true=y_test,
    y_score=forensic_test_proba,
    base_df=test_df,
    model_name="RQ1_Forensic_XGB",
)

lift_rq2_bovw_k = compute_lift_table(
    y_true=y_test,
    y_score=bovw_k_test_proba,
    base_df=test_df,
    model_name="RQ2_BoVW_plus_Kmeans_XGB",
)

lift_rq3_bovw_raw = compute_lift_table(
    y_true=y_test,
    y_score=bovw_test_proba,
    base_df=test_df,
    model_name="RQ3_BoVW_raw_XGB",
)

lift_rq3_bovw_pca = compute_lift_table(
    y_true=y_test,
    y_score=pca_test_proba,
    base_df=test_df,
    model_name="RQ3_BoVW_PCA_XGB",
)

lift_rq4_cnn = compute_lift_table(
    y_true=y_test,
    y_score=cnn_test_proba,
    base_df=test_df,
    model_name="RQ4_CNN",
)

lift_rq5_ensemble = compute_lift_table(
    y_true=y_test,
    y_score=ens_test_proba,
    base_df=test_df,
    model_name="RQ5_Ensemble_XGB",
)

# Quick peek at one of them
display(lift_rq3_bovw_pca)


Test length: 11520
  y_test: 11520
  forensic_test_proba: 11520
  bovw_k_test_proba: 11520
  bovw_test_proba: 11520
  pca_test_proba: 11520
  cnn_test_proba: 11520
  ens_test_proba: 11520
[LIFT] Saved RQ1_Forensic_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ1_Forensic_XGB_test.csv
[LIFT] Saved RQ2_BoVW_plus_Kmeans_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ2_BoVW_plus_Kmeans_XGB_test.csv
[LIFT] Saved RQ3_BoVW_raw_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ3_BoVW_raw_XGB_test.csv
[LIFT] Saved RQ3_BoVW_PCA_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ3_BoVW_PCA_XGB_test.csv
[LIFT] Saved RQ4_CNN lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ4_CNN_test.csv
[LIFT] Saved RQ5_Ensemble_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ5_Ensemble_XGB_test.csv


,decile,n,n_fake,n_real,pct_fake,pct_real,cum_n,cum_fake,cum_fake_rate,pop_frac,...,n_fake_postproc,n_fake_splicing,n_fake_stargan_tiles,n_fake_swap_like,pct_fake_copy_move,pct_fake_inpaint,pct_fake_postproc,pct_fake_splicing,pct_fake_stargan_tiles,pct_fake_swap_like
0,1,1152,937,215,0.813368,0.186632,1152,937,0.162674,0.1,...,158,38,623,31,0.044824,0.048026,0.168623,0.040555,0.664888,0.033084
1,2,1152,712,440,0.618056,0.381944,2304,1649,0.286285,0.2,...,144,88,195,118,0.130618,0.103933,0.202247,0.123596,0.273876,0.165730
2,3,1152,608,544,0.527778,0.472222,3456,2257,0.391840,0.3,...,100,115,61,140,0.161184,0.154605,0.164474,0.189145,0.100329,0.230263
3,4,1152,609,543,0.528646,0.471354,4608,2866,0.497569,0.4,...,104,110,39,140,0.178982,0.175698,0.170772,0.180624,0.064039,0.229885
4,5,1152,557,595,0.483507,0.516493,5760,3423,0.594271,0.5,...,93,127,13,127,0.204668,0.149013,0.166966,0.228007,0.023339,0.228007
5,6,1152,520,632,0.451389,0.548611,6912,3943,0.684549,0.6,...,83,107,16,106,0.203846,0.196154,0.159615,0.205769,0.030769,0.203846
6,7,1152,492,660,0.427083,0.572917,8064,4435,0.769965,0.7,...,71,100,4,93,0.219512,0.235772,0.144309,0.203252,0.008130,0.189024
7,8,1152,487,665,0.422743,0.577257,9216,4922,0.854514,0.8,...,78,101,7,88,0.229979,0.207392,0.160164,0.207392,0.014374,0.180698
8,9,1152,455,697,0.394965,0.605035,10368,5377,0.933507,0.9,...,71,108,2,67,0.224176,0.230769,0.156044,0.237363,0.004396,0.147253
9,10,1152,383,769,0.332465,0.667535,11520,5760,1.000000,1.0,...,58,66,0,50,0.198433,0.347258,0.151436,0.172324,0.000000,0.130548


In [33]:
# === Zip StarGAN-v2 'data' folder and move to Drive ===
from pathlib import Path
from datetime import datetime
import shutil, sys

SRC = Path("/content/stargan-v2/explainability")
DEST_DIR = Path("/content/drive/MyDrive/deepfake_project/output/explainability")

# Safety checks
if not SRC.exists():
    raise FileNotFoundError(f"Source folder not found: {SRC}")
if not any(SRC.rglob("*")):
    raise RuntimeError(f"Source folder is empty: {SRC}")

DEST_DIR.mkdir(parents=True, exist_ok=True)

# Timestamped archive name
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
archive_base = Path(f"/content/stargan-v2_explainability-{stamp}")  # no extension for make_archive

print(f"Creating ZIP archive from: {SRC}")
# Create ZIP (you can switch 'zip' -> 'gztar' for .tar.gz if preferred)
zip_path_str = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=SRC.parent,   # /content/stargan-v2
    base_dir=SRC.name      # data
)
archive_path = Path(zip_path_str)

# Move to Drive
final_path = DEST_DIR / archive_path.name
print(f"Moving archive to: {final_path}")
shutil.move(str(archive_path), str(final_path))

# Report
size_gb = final_path.stat().st_size / (1024**3)
print(f"✅ Archive ready: {final_path}")
print(f"   Size: {size_gb:.2f} GB")


Creating ZIP archive from: /content/stargan-v2/explainability
Moving archive to: /content/drive/MyDrive/deepfake_project/output/explainability/stargan-v2_explainability-20251229-220830.zip
✅ Archive ready: /content/drive/MyDrive/deepfake_project/output/explainability/stargan-v2_explainability-20251229-220830.zip
   Size: 0.02 GB


In [35]:
print("Completed")

Completed
